# Cocoa Contamination Hackathon — YOLO11 (T4)

Self-contained training + inference notebook for the Zindi IndabaX SA Cocoa Contamination Hackathon.

**Metric:** mAP@IoU0.5 &nbsp;|&nbsp; **Limits:** single T4, &le;9h train, &le;3h infer &nbsp;|&nbsp; **Edge:** ONNX export.

### Before running
1. Settings &rarr; **Accelerator = GPU T4 x1**, **Internet = On**.
2. Add the data. Either:
   - Upload `dataset.zip` (the ~9.4 GB images) as a Kaggle Dataset and attach it, **or**
   - Let cell 2 download it from the public URL.
3. Run all cells. Final output: `submission.csv` + `best.onnx`.

In [11]:
# 1. Install deps
!pip -q install "ultralytics>=8.3.0" onnx onnxslim onnxruntime
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

ultralytics 8.4.90 | CUDA False 


In [12]:
# 2. Locate or download the dataset  (works on Kaggle AND locally)
#    The archive already contains YOLO labels: images/{train,test}/ and labels/train/
import os, glob, zipfile, urllib.request
from pathlib import Path

# Pick a working dir depending on environment
# (use env var, not Path('/kaggle') — on Windows that can match a stray C:\kaggle folder)
ON_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None or os.path.isdir('/kaggle/input')

def has_dataset(root: Path) -> bool:
    return (root / 'labels' / 'train').is_dir() and (root / 'images' / 'train').is_dir()

def valid_zip(p: Path) -> bool:
    try:
        return p.exists() and zipfile.is_zipfile(p)
    except OSError:
        return False

# Robustly locate a data dir that already has extracted data or a valid zip.
# Searches cwd and up to 3 parent levels (kernel cwd may differ from project root).
def locate_data() -> Path:
    roots = []
    here = Path.cwd()
    for base in [here, *here.parents[:3]]:
        roots += [base / 'data', base]
    for r in roots:
        if has_dataset(r) or valid_zip(r / 'dataset.zip'):
            return r
    return here / 'data'   # default target for download

DATA = Path('/kaggle/working/data') if ON_KAGGLE else locate_data()
DATA.mkdir(parents=True, exist_ok=True)
print('cwd:', Path.cwd(), '| DATA:', DATA.resolve())

# 1) Already extracted? (local DATA, or an attached Kaggle dataset)
candidates = [DATA]
candidates += [Path(p).parents[1] for p in glob.glob('/kaggle/input/**/labels/train', recursive=True)]
SRC = next((c for c in candidates if has_dataset(c)), None)

if SRC is not None:
    print('Dataset already extracted at', SRC.resolve(), '— skipping unzip')
else:
    # 2) Find a valid zip: attached, or local DATA/dataset.zip
    zip_candidates = (glob.glob('/kaggle/input/**/dataset.zip', recursive=True)
                      + [str(DATA / 'dataset.zip')])
    zip_path = next((Path(z) for z in zip_candidates if valid_zip(Path(z))), DATA / 'dataset.zip')

    if not valid_zip(zip_path):
        print('No valid dataset.zip found — downloading (~9.4 GB)...')
        urllib.request.urlretrieve('https://storage.googleapis.com/cocoa/dataset.zip', zip_path)

    print('Extracting', zip_path, '...')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA)
    SRC = DATA

print('train imgs:', len(glob.glob(str(SRC/'images/train/*'))),
      '| test imgs:', len(glob.glob(str(SRC/'images/test/*'))),
      '| labels:', len(glob.glob(str(SRC/'labels/train/*.txt'))))

cwd: c:\Users\USER PC\Desktop\New folder | DATA: C:\Users\USER PC\Desktop\New folder\data
Dataset already extracted at C:\Users\USER PC\Desktop\New folder\data — skipping unzip
train imgs: 5529 | test imgs: 1626 | labels: 5529


In [13]:
# 3. Build a seeded 85/15 train/val split (uses provided YOLO labels)
import os, random, shutil
from pathlib import Path

CLASSES = ['anthracnose', 'cssvd', 'healthy']   # index order = provided labels (0,1,2)
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('.')
OUT = WORK / 'yolo'
SRC_IMG, SRC_LBL = SRC/'images/train', SRC/'labels/train'
VAL_FRACTION, SEED = 0.15, 42

def find_image(stem):
    for ext in ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'):
        p = SRC_IMG / f'{stem}{ext}'
        if p.exists():
            return p
    return None

def link_or_copy(src, dst):
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

stems = sorted(p.stem for p in SRC_LBL.glob('*.txt'))
random.Random(SEED).shuffle(stems)
n_val = int(len(stems) * VAL_FRACTION)
splits = {'val': stems[:n_val], 'train': stems[n_val:]}

for sub in ('images', 'labels'):
    for split in ('train', 'val'):
        (OUT/sub/split).mkdir(parents=True, exist_ok=True)

counts = {'train': 0, 'val': 0}
for split, ids in splits.items():
    for stem in ids:
        img = find_image(stem)
        lbl = SRC_LBL / f'{stem}.txt'
        if img is None or not lbl.exists():
            continue
        link_or_copy(img, OUT/'images'/split/img.name)
        link_or_copy(lbl, OUT/'labels'/split/lbl.name)
        counts[split] += 1

(OUT/'cocoa.yaml').write_text(
    f'path: {OUT.as_posix()}\ntrain: images/train\nval: images/val\n'
    f'nc: {len(CLASSES)}\nnames: {CLASSES}\n')
print('train', counts['train'], '| val', counts['val'], '| classes', CLASSES)

train 4700 | val 829 | classes ['anthracnose', 'cssvd', 'healthy']


In [ ]:
# 4. Train (yolo11s @ 640). Fits the T4 / 9h budget. Auto-exports ONNX.
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'   # prefer GPU by default
print('Training on', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
MODEL, IMGSZ, EPOCHS, NAME = 'yolo11s.pt', 640, 100, 'cocoa_yolo11s'

model = YOLO(MODEL)
results = model.train(
    data=str(OUT/'cocoa.yaml'), imgsz=IMGSZ, epochs=EPOCHS, batch=-1,
    patience=25, project=str(WORK / 'runs'), name=NAME, seed=42, device=DEVICE,
    cache='disk', cos_lr=True, close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, flipud=0.2,
    mosaic=1.0, mixup=0.1, scale=0.5, amp=True, plots=True,
)
best = Path(results.save_dir) / 'weights' / 'best.pt'

print('best weights:', best)print('ONNX exported next to best.pt')
YOLO(str(best)).export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)

Ultralytics 8.4.90  Python-3.11.9 torch-2.7.1+cpu CPU (AMD Ryzen 9 9900X 12-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=yolo\cocoa.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cocoa_yolo11s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=25

In [ ]:
# 5. Inference -> submission.csv (format: Image_ID,class,confidence,ymin,xmin,ymax,xmax)
import csv
from ultralytics import YOLO

CONF, MAX_DET, TTA = 0.05, 100, True
TEST_CSV = glob.glob(str(SRC/'..'/'Test.csv')) + glob.glob('/kaggle/input/**/Test.csv', recursive=True) + glob.glob(str(DATA/'Test.csv'))
TEST_CSV = next((p for p in TEST_CSV if Path(p).exists()), None)
assert TEST_CSV, 'Test.csv not found — attach it or place in data/'
IMG_DIR = SRC/'images/test'

with open(TEST_CSV, newline='') as f:
    test_ids = [r['Image_ID'] for r in csv.DictReader(f)]

model = YOLO(str(best))
rows, seen = [], set()
for img_id in test_ids:
    path = IMG_DIR / img_id
    if not path.exists():
        continue
    r = model.predict(source=str(path), imgsz=IMGSZ, conf=CONF, max_det=MAX_DET,
                      augment=TTA, device=DEVICE, verbose=False)[0]
    for box in r.boxes:
        cls = CLASSES[int(box.cls)]
        conf = float(box.conf)
        x1, y1, x2, y2 = (float(v) for v in box.xyxy[0])
        rows.append([img_id, cls, round(conf, 6), round(y1), round(x1), round(y2), round(x2)])
        seen.add(img_id)

for img_id in test_ids:            # placeholder for zero-detection images
    if img_id not in seen:
        rows.append([img_id, 'healthy', 0.01, 0, 0, 1, 1])

sub_path = WORK / 'submission.csv'
with open(sub_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax'])
    w.writerows(rows)
print('wrote', len(rows), 'rows for', len(test_ids), 'images ->', sub_path)